In [1]:
# Import required libraries
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, lit, current_date

# Create Spark session
spark = SparkSession.builder.getOrCreate()

# Constants for KQL source
cluster_uri = "https://trd-kzw6ts318kufakm1np.z0.kusto.fabric.microsoft.com"
database_name = "streaming_betting_data"
lakehouse_output_path = "Files/cdp_bronze/kafka_eventstream/raw_stream/"  # Updated to Delta Lake Tables layer

# List of tables to export
tables = ["bet_results", "betting_events", "live_odds_updates", "match_metadata"]

# Time filter: last 1 day
kql_time_filter = " | where ingestion_time() > ago(1d)"

# Loop through tables and export
for table_name in tables:
    print(f"📥 Reading from KQL table: {table_name}")
    
    kql_query = f"{table_name}{kql_time_filter}"
    
    # Read from KQL
    df = spark.read.format("com.microsoft.kusto.spark.datasource") \
        .option("kustoCluster", cluster_uri) \
        .option("kustoDatabase", database_name) \
        .option("kustoQuery", kql_query) \
        .option("accessToken", mssparkutils.credentials.getToken(cluster_uri)) \
        .load()

    # Add event_date column for partitioning
    if "ingestion_time" in df.columns:
        df = df.withColumn("event_date", to_date(col("ingestion_time")))
    else:
        # Optional: fallback if your schema doesn't have ingestion_time column
        df = df.withColumn("event_date", to_date(lit(current_date())))

    print(f"📤 Writing to Lakehouse path: {lakehouse_output_path}{table_name}")
    
    # Write to Silver layer in Delta format, partitioned by date
    df.write.mode("append").partitionBy("event_date").format("delta").save(f"{lakehouse_output_path}{table_name}")

    print(f"✅ {table_name} exported successfully.")


StatementMeta(, a588bd27-ae1d-4ed0-b4e2-d220dc499dfd, 3, Finished, Available, Finished)

📥 Reading from KQL table: bet_results
📤 Writing to Lakehouse path: Files/cdp_bronze/kafka_eventstream/raw_stream/bet_results
✅ bet_results exported successfully.
📥 Reading from KQL table: betting_events
📤 Writing to Lakehouse path: Files/cdp_bronze/kafka_eventstream/raw_stream/betting_events
✅ betting_events exported successfully.
📥 Reading from KQL table: live_odds_updates
📤 Writing to Lakehouse path: Files/cdp_bronze/kafka_eventstream/raw_stream/live_odds_updates
✅ live_odds_updates exported successfully.
📥 Reading from KQL table: match_metadata
📤 Writing to Lakehouse path: Files/cdp_bronze/kafka_eventstream/raw_stream/match_metadata
✅ match_metadata exported successfully.
